# Multi-Quantile Intervals: Inventory Service Levels
# 多分位区间：库存服务水平

Scenario: inventory teams need multiple uncertainty bands to choose safety stock for 50%, 80%, 90%, and 95% service levels.

场景：库存团队需要多个不确定性区间，为 50%、80%、90%、95% 服务水平选择安全库存。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
from PipelineTS.pipeline import ModelPipeline, SmartRouter
from PipelineTS.metrics import picp, pinaw, winkler_score

data = make_retail_demand(n_days=240, n_stores=1).drop(columns=["store_id"])
train, valid = data.iloc[:-28].copy(), data.iloc[-28:].copy()
horizon = 28

In [ ]:
pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=14,
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    quantile=0.9,
    cv=3,
    random_forest__n_estimators=100,
    extra_forest__n_estimators=100,
)
pipe.fit(train, valid_data=valid)

point_and_90 = pipe.predict(horizon)
multi_q = pipe.predict_quantiles(horizon, levels=[0.5, 0.8, 0.9, 0.95])
multi_q.head()

In [ ]:
y_true = valid["sales"].values[:horizon]
rows = []
for level in [0.5, 0.8, 0.9, 0.95]:
    label = f"{level:.2f}".rstrip("0").rstrip(".")
    lo = multi_q[f"sales_q{label}_lower"].values[:horizon]
    hi = multi_q[f"sales_q{label}_upper"].values[:horizon]
    rows.append({
        "coverage_level": level,
        "PICP": picp(y_true, lo, hi),
        "PINAW": pinaw(y_true, lo, hi),
        "Winkler": winkler_score(y_true, lo, hi, alpha=1-level),
    })
pd.DataFrame(rows)

In [ ]:
router = SmartRouter(
    time_col="date",
    target_col="sales",
    preset="fast",
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    quantile=0.9,
    cv=3,
    time_limit=60,
)
router.fit(train, valid_data=valid)
router.predict_quantiles(horizon, levels=[0.8, 0.9, 0.95]).head()

In [ ]:
from PipelineTS.plot import plot_forecast

plot_forecast(train, point_and_90, time_col="date", target_col="sales", history_tail=90, lang="zh")